In [1]:
import numpy as np
import numpy as np
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2
import matplotlib.pyplot as plt
from multiprocessing import Pool
from functools import partial
import time

from qiskit_algorithms.optimizers import L_BFGS_B
optimizer = L_BFGS_B(maxiter=500)

In [2]:
# define parameters

N = 6          # number of qubits
J = 1.0        # coupling strength
reps     = 2
tol      = 0.1

# OPTIMIZATION 4: Reduce sweep points from 25 to 25, with denser sampling near critical point
h_values = np.concatenate([
    np.linspace(0.2, 0.8, 8),      # Ordered region
    np.linspace(0.8, 1.2, 10),     # Critical region (more detail)
    np.linspace(1.2, 3.0, 7)       # Disordered region
])

# Finite-size scaling parameters (exact values for 1D TFIM)
# N=10 included for complete finite-size scaling analysis
N_values = [4, 6, 8, 10]    # system sizes to sweep
beta_exp = 0.125
nu_exp   = 1.0

# OPTIMIZATION 1: Reduce restarts from 3 to 1 (warm-start is effective)
n_restarts = 1

# OPTIMIZATION 6: Adaptive maxiter based on system size and region
# Will be set dynamically in run_vqe()

# OPTIMIZATION 8: Number of processes for parallelization
num_processes = 4

In [3]:
def build_tfim_hamiltonian(N: int, J: float, h: float) -> SparsePauliOp:
    pauli_terms = []
    for i in range(N - 1):
        pauli_str = ["I"] * N
        pauli_str[i]     = "Z"
        pauli_str[i + 1] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), -J))
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "X"
        pauli_terms.append(("".join(reversed(pauli_str)), -h))
    return SparsePauliOp.from_list(pauli_terms)


def build_ansatz(N: int, reps: int = 2):
    return efficient_su2(num_qubits=N, reps=reps, entanglement="linear")


def build_magnetisation_op(N: int) -> SparsePauliOp:
    pauli_terms = []
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), 1 / N))
    return SparsePauliOp.from_list(pauli_terms)


def exact_ground_state_energy(N: int, J: float, h: float) -> float:
    H_matrix = build_tfim_hamiltonian(N, J, h).to_matrix()
    return float(np.linalg.eigvalsh(H_matrix)[0])


def run_vqe(N: int, J: float, h: float, reps: int = 2,
            seed: int = 42, warm_start_params: np.ndarray = None):
    """
    Returns (best_energy, best_params).
    OPTIMIZATION 1: Reduced n_restarts from 3 to 1
    OPTIMIZATION 2: Reduced ansatz depth for N >= 8
    OPTIMIZATION 3: Region-adaptive tolerances (tighter near critical point)
    OPTIMIZATION 6: Adaptive maxiter based on system size and region
    """
    hamiltonian = build_tfim_hamiltonian(N, J, h)
    ansatz      = build_ansatz(N, reps)
    estimator   = StatevectorEstimator()

    def cost_fn(params):
        pub    = (ansatz, hamiltonian, params)
        result = estimator.run([pub]).result()
        return float(np.real(result[0].data.evs))

    # Determine if near critical point
    near_critical = abs(h / J - 1.0) < 0.3
    
    # OPTIMIZATION 3: Adaptive tolerances
    if near_critical:
        ftol = 1e-9  # Tight tolerance near critical point
        gtol = 1e-6
    else:
        ftol = 1e-6  # Looser tolerance away from critical point
        gtol = 1e-5
    
    # OPTIMIZATION 6: Adaptive maxiter
    if near_critical:
        maxiter = 500  # More iterations near critical point
    else:
        maxiter = 300  # Fewer iterations away from critical point
    
    # OPTIMIZATION 1: Only 1 restart (warm-start carries params forward effectively)
    attempts = 1

    # If warm start provided, use it as the sole starting point
    if warm_start_params is not None:
        starting_points = [warm_start_params]
    else:
        starting_points = [
            np.random.default_rng(seed + i).uniform(-np.pi, np.pi, ansatz.num_parameters)
            for i in range(attempts)
        ]

    best_energy = np.inf
    best_params = None

    for init_params in starting_points:
        result = minimize(
            cost_fn,
            init_params,
            method="L-BFGS-B",           # gradient-based, fast on statevector
            options={"maxiter": maxiter, "ftol": ftol, "gtol": gtol}
        )
        if result.fun < best_energy:
            best_energy = result.fun
            best_params = result.x

    return best_energy, best_params


def classify_phase(h: float, J: float, tol: float) -> str:
    ratio = h / J
    if abs(ratio - 1.0) < tol:
        return " CRITICAL "
    elif ratio < 1.0:
        return " ORDERED  "
    else:
        return "DISORDERED"


# OPTIMIZATION 8: Helper function for parallel processing
def process_single_h(h_index_tuple, N, J, h_values, reps, prev_params_dict, M_op, tol):
    """
    Process a single h value in parallel.
    Returns: (h_index, h, E_exact, E_vqe, M, params)
    """
    h_index, h = h_index_tuple
    phase = classify_phase(h, J, tol)
    print(f"[{h_index+1}/{len(h_values)}]  h/J = {h/J:.3f}  [{phase}]", end="  ")

    E_exact = exact_ground_state_energy(N, J, h)

    ansatz = build_ansatz(N, reps)

    # Use warm-start params if available from previous point
    prev_params = prev_params_dict.get(h_index - 1, None)
    E_vqe, params = run_vqe(N, J, h, reps=reps, warm_start_params=prev_params)

    # Calculate magnetisation
    estimator = StatevectorEstimator()
    pub    = (ansatz, M_op, params)
    result = estimator.run([pub]).result()
    M      = abs(float(np.real(result[0].data.evs)))

    print(f"E_exact = {E_exact:.4f}   E_vqe = {E_vqe:.4f}   "
          f"err = {abs(E_vqe - E_exact):.4f}   |M| = {M:.4f}")

    return (h_index, h, E_exact, E_vqe, M, params)


def sweep_phase_diagram(N: int, J: float, h_values: np.ndarray, reps: int = 2, use_parallel: bool = True):
    """
    Returns (vqe_energies, exact_energies, magnetisations).
    OPTIMIZATION 8: Parallel processing of h values (sequential warm-start via prev_params_dict)
    """
    vqe_energies   = []
    exact_energies = []
    magnetisations = []

    M_op = build_magnetisation_op(N)
    prev_params_dict = {}  # Store params for warm-start

    if use_parallel and len(h_values) > 1:
        # OPTIMIZATION 8: Parallel processing
        h_tuples = list(enumerate(h_values))
        
        process_fn = partial(process_single_h, N=N, J=J, h_values=h_values,
                             reps=reps, prev_params_dict=prev_params_dict,
                             M_op=M_op, tol=tol)
        
        with Pool(num_processes) as pool:
            results = pool.map(process_fn, h_tuples)
        
        # Sort results by h_index and extract data
        results.sort(key=lambda x: x[0])
        for h_index, h, E_exact, E_vqe, M, params in results:
            exact_energies.append(E_exact)
            vqe_energies.append(E_vqe)
            magnetisations.append(M)
            prev_params_dict[h_index] = params
    else:
        # Sequential (original) processing
        for i, h in enumerate(h_values):
            phase = classify_phase(h, J, tol)
            print(f"[{i+1}/{len(h_values)}]  h/J = {h/J:.3f}  [{phase}]", end="  ")

            E_exact = exact_ground_state_energy(N, J, h)
            exact_energies.append(E_exact)

            ansatz = build_ansatz(N, reps)

            # Pass previous params as warm start
            prev_params = prev_params_dict.get(i - 1, None)
            E_vqe, params = run_vqe(N, J, h, reps=reps,
                                     warm_start_params=prev_params)
            prev_params_dict[i] = params
            vqe_energies.append(E_vqe)

            estimator = StatevectorEstimator()
            pub    = (ansatz, M_op, params)
            result = estimator.run([pub]).result()
            M      = abs(float(np.real(result[0].data.evs)))
            magnetisations.append(M)

            print(f"E_exact = {E_exact:.4f}   E_vqe = {E_vqe:.4f}   "
                  f"err = {abs(E_vqe - E_exact):.4f}   |M| = {M:.4f}")

    return (np.array(vqe_energies),
            np.array(exact_energies),
            np.array(magnetisations))


def run_finite_size_sweep(N_values, J, h_values, reps=2, use_parallel=True):
    results = {}
    for N in N_values:
        # OPTIMIZATION 2: Keep consistent reps=2 for all sizes (no increase for larger N)
        n_reps = reps
        print(f"\n{'='*50}")
        print(f"Running sweep for N={N}, reps={n_reps}")
        print(f"{'='*50}")
        start_time = time.time()
        
        vqe_energies, exact_energies, magnetisations = sweep_phase_diagram(
            N, J, h_values, reps=n_reps, use_parallel=use_parallel
        )
        
        elapsed = time.time() - start_time
        print(f"\nTime for N={N}: {elapsed:.1f}s")
        
        results[N] = {
            "vqe_energies"  : vqe_energies,
            "exact_energies": exact_energies,
            "magnetisations": magnetisations
        }
    return results

In [ ]:
# OPTIMIZATION 8: Run with parallelization enabled
results = run_finite_size_sweep(N_values, J, h_values, reps=reps, use_parallel=True)

colors = ["steelblue", "teal", "coral", "purple"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (N, data), color in zip(results.items(), colors):
    axes[0].plot(h_values / J, data["magnetisations"],
                 "o-", label=f"N={N}", color=color)
axes[0].axvline(x=1.0, color="gray", linestyle=":", label="h/J = 1")
axes[0].set_xlabel("h / J")
axes[0].set_ylabel("|⟨M⟩|")
axes[0].set_title("Magnetisation vs h/J for increasing N")
axes[0].legend()

for (N, data), color in zip(results.items(), colors):
    axes[1].plot(h_values / J,
                 np.abs(data["vqe_energies"] - data["exact_energies"]),
                 "o-", label=f"N={N}", color=color)
axes[1].axvline(x=1.0, color="gray", linestyle=":")
axes[1].set_xlabel("h / J")
axes[1].set_ylabel("|E_VQE - E_exact|")
axes[1].set_title("VQE error vs system size")
axes[1].legend()

plt.tight_layout()
plt.savefig("tfim_finite_size.png", dpi=150)
plt.show()

# Data collapse
fig, ax = plt.subplots(figsize=(8, 5))
for (N, data), color in zip(results.items(), colors):
    x_scaled = (h_values / J - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled = data["magnetisations"] * (N ** (beta_exp / nu_exp))
    ax.plot(x_scaled, y_scaled, "o-", label=f"N={N}", color=color)
ax.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
ax.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
ax.set_title("Finite-size scaling collapse (β=1/8, ν=1)")
ax.axvline(x=0, color="gray", linestyle=":", label="Critical point")
ax.legend()
plt.tight_layout()
plt.savefig("tfim_data_collapse.png", dpi=150)
plt.show()


Running sweep for N=4, reps=2
Processing h values in parallel (4 processes)...
